In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from functions import (load_month, load_month_sa, load_month_pen, mapd_clean_merge)
import re 
%pip install xlrd>=2.0.1
!pip install openpyxl
!pip -q install rpy2
%load_ext rpy2.ipython
    
monthlist = [f"{m:02d}" for m in range(1, 12)]
y = 2012


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Plan (Enrollment and Contract) Data

In [2]:
# Load and combine monthly data
plan_data = pd.concat([load_month(m, y) for m in monthlist], ignore_index=True)

# Sort data
plan_data = plan_data.sort_values(['contractid', 'planid', 'state', 'county', 'month'])

# Fill fips by state/county groups
plan_data['fips'] = plan_data.groupby(['state', 'county'])['fips'].transform(
    lambda x: x.ffill().bfill()
)

# Fill plan-level attributes
plan_cols = ['plan_type', 'partd', 'snp', 'eghp', 'plan_name']
for col in plan_cols:
    plan_data[col] = plan_data.groupby(['contractid', 'planid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Fill contract-level attributes
contract_cols = ['org_type', 'org_name', 'org_marketing_name', 'parent_org']
for col in contract_cols:
    plan_data[col] = plan_data.groupby(['contractid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Sort for aggregation
plan_data = plan_data.sort_values(['contractid', 'planid', 'fips', 'year', 'month'])

# Aggregate to yearly level
def agg_plan_year(group):
    nonmiss = group['enrollment'].notna()
    n = nonmiss.sum()
    enroll_vals = group.loc[nonmiss, 'enrollment']
    
    return pd.Series({
        'n_nonmiss': n,
        'avg_enrollment': enroll_vals.mean() if n > 0 else np.nan,
        'first_enrollment': enroll_vals.iloc[0] if n > 0 else np.nan,
        'last_enrollment': enroll_vals.iloc[-1] if n > 0 else np.nan,
        'state': group['state'].iloc[-1],
        'county': group['county'].iloc[-1],
        'org_type': group['org_type'].iloc[-1],
        'plan_type': group['plan_type'].iloc[-1],
        'partd': group['partd'].iloc[-1],
        'snp': group['snp'].iloc[-1],
        'eghp': group['eghp'].iloc[-1],
        'org_name': group['org_name'].iloc[-1],
        'org_marketing_name': group['org_marketing_name'].iloc[-1],
        'plan_name': group['plan_name'].iloc[-1],
        'parent_org': group['parent_org'].iloc[-1],
        'contract_date': group['contract_date'].iloc[-1]
    })

plan_data_2012 = plan_data.groupby(['contractid', 'planid', 'fips', 'year']).apply(agg_plan_year).reset_index()
print(f"Plan data shape: {plan_data_2012.shape}")

/tmp/ipykernel_220508/2829393326.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()


Plan data shape: (2177642, 20)


/tmp/ipykernel_220508/2829393326.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  plan_data_2012 = plan_data.groupby(['contractid', 'planid', 'fips', 'year']).apply(agg_plan_year).reset_index()


## Service Area Data

In [3]:
import sys
sys.executable

# Load and combine monthly service area data
service_year = pd.concat([load_month_sa(m, y) for m in monthlist], ignore_index=True)

# Sort for stable fills
service_year = service_year.sort_values(['contractid', 'fips', 'state', 'county', 'month'])

# Fill fips by state/county groups
service_year['fips'] = service_year.groupby(['state', 'county'])['fips'].transform(
    lambda x: x.ffill().bfill()
)

# Fill contract-level attributes
contract_cols_sa = ['plan_type', 'partial', 'eghp', 'org_type', 'org_name']
for col in contract_cols_sa:
    service_year[col] = service_year.groupby(['contractid'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Collapse to yearly: one row per contract x county (fips) x year
service_year = service_year.sort_values(['contractid', 'fips', 'year', 'month'])

service_data_2012 = service_year.groupby(['contractid', 'fips', 'year']).agg({
    'state': 'last',
    'county': 'last',
    'org_name': 'last',
    'org_type': 'last',
    'plan_type': 'last',
    'partial': 'last',
    'eghp': 'last',
    'ssa': 'last',
    'notes': 'last'
}).reset_index()

print(f"Service area data shape: {service_data_2012.shape}")

/tmp/ipykernel_220508/1393515547.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()
/tmp/ipykernel_220508/1393515547.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()


Service area data shape: (374145, 12)


## Plan Characteristics Data

In [4]:
MA_COLUMNS_2011 = [
    "state","county","org_name","plan_name","plan_type","premium","partd_deductible",
    "drug_type","gap_coverage","drug_type_detail","demo_type","contractid",
    "planid","segmentid","moop"
]

MA_DTYPES_2011 = {col: "string" for col in MA_COLUMNS_2011}


# -------------------------------
# MAPD Landscape Column Definitions
# -------------------------------

MAPD_COLUMNS_2011 = [
    "state","county","org_name","plan_name","contractid","planid","segmentid",
    "org_type","plan_type","snp","snp_type","benefit_type","below_benchmark",
    "national_pdp","partd_rein_demo","partd_rein_demo_type","premium_partc",
    "premium_partd_basic","premium_partd_supp","premium_partd_total",
    "partd_assist_full","partd_assist_75","partd_assist_50","partd_assist_25",
    "partd_deductible","deductible_exclusions","increase_coverage_limit",
    "gap_coverage","gap_coverage_type"
]

MAPD_DTYPES_2011 = {col: "string" for col in MAPD_COLUMNS_2011}

In [5]:
import pandas as pd
from pathlib import Path
from functions import MA_DATA_DIR

y = 2012
base = MA_DATA_DIR / "ma" / "landscape" / "Extracted Data"

# ---- 1) Load MA landscape (2012) ----
ma_path_a = base / "2012LandscapeSourceData_MA_3_08_12_AtoM.csv"
ma_path_b = base / "2012LandscapeSourceData_MA_3_08_12_NtoW.csv"

ma_data_a = pd.read_csv(
    ma_path_a,
    skiprows=5,
    header=None,
    names=MA_COLUMNS_2011,      # reuse your 2010 definitions
    dtype=MA_DTYPES_2011,
    encoding="latin1",
    encoding_errors="replace",
    low_memory=False,
)
ma_data_b = pd.read_csv(
    ma_path_b,
    skiprows=5,
    header=None,
    names=MA_COLUMNS_2011,
    dtype=MA_DTYPES_2011,
    encoding="latin1",
    encoding_errors="replace",
    low_memory=False,
)
ma_data = pd.concat([ma_data_a, ma_data_b], ignore_index=True)

# ---- 2) Load Part D landscape (2011) ----
partcd_dir = base / "PartCD" / str(y)
xls_candidates = sorted(partcd_dir.glob("*.xls")) + sorted(partcd_dir.glob("*.xlsx"))
if len(xls_candidates) == 0:
    raise FileNotFoundError(f"No Part D workbook found in {partcd_dir}")

mapd_path = xls_candidates[0]
print("Using Part D file:", mapd_path)

xls = pd.ExcelFile(mapd_path, engine="xlrd")
print("Sheets:", xls.sheet_names)

# common sheet names; otherwise take first two
if "Alabama to Montana" in xls.sheet_names and "Nebraska to Wyoming" in xls.sheet_names:
    s1, s2 = "Alabama to Montana", "Nebraska to Wyoming"
else:
    s1, s2 = xls.sheet_names[0], xls.sheet_names[1]

mapd_data_a = pd.read_excel(
    mapd_path,
    engine="xlrd",
    sheet_name=s1,
    skiprows=4,
    header=None,
    names=MAPD_COLUMNS_2011,
    dtype=MAPD_DTYPES_2011,
)
mapd_data_b = pd.read_excel(
    mapd_path,
    engine="xlrd",
    sheet_name=s2,
    skiprows=4,
    header=None,
    names=MAPD_COLUMNS_2011,
    dtype=MAPD_DTYPES_2011,
)
mapd_data = pd.concat([mapd_data_a, mapd_data_b], ignore_index=True)

print("ma_data shape:", ma_data.shape)
print("mapd_data shape:", mapd_data.shape)

Using Part D file: /home/gdanzil/econ470/a0/work/ma-data/ma/landscape/Extracted Data/PartCD/2012/Medicare Part D 2012 Plan Report 09-08-11.xls
Sheets: ['Alabama to Montana', 'Nebraska to Wyoming', 'Important Notes']
ma_data shape: (43561, 15)
mapd_data shape: (39695, 29)


In [6]:
import pandas as pd

def mapd_clean_merge_norecast(ma_data: pd.DataFrame, mapd_data: pd.DataFrame, y: int) -> pd.DataFrame:
    # --- force merge keys to string ---
    for df in [ma_data, mapd_data]:
        df["contractid"] = df["contractid"].astype("string")
        df["state"] = df["state"].astype("string")
        df["county"] = df["county"].astype("string")
        df["planid"] = df["planid"].astype("string").str.replace(r"\.0$", "", regex=True)

        # (optional but helpful) strip whitespace
        df["contractid"] = df["contractid"].str.strip()
        df["state"] = df["state"].str.strip()
        df["county"] = df["county"].str.strip()
        df["planid"] = df["planid"].str.strip()

    # Tidy MA-only data
    ma = ma_data[["contractid", "planid", "state", "county", "premium"]].copy()
    ma = ma.sort_values(["contractid", "planid", "state", "county"])
    ma["premium"] = ma.groupby(["contractid", "planid", "state", "county"])["premium"].ffill()
    ma = ma.drop_duplicates(subset=["contractid", "planid", "state", "county"], keep="first")

    # Tidy MA-PD data (IMPORTANT: do NOT coerce planid to numeric)
    md = mapd_data[
        ["contractid", "planid", "state", "county",
         "premium_partc", "premium_partd_basic", "premium_partd_supp",
         "premium_partd_total", "partd_deductible"]
    ].copy()

    md = md.sort_values(["contractid", "planid", "state", "county"])
    fill_cols = ["premium_partc", "premium_partd_basic", "premium_partd_supp",
                 "premium_partd_total", "partd_deductible"]
    md[fill_cols] = md.groupby(["contractid", "planid", "state", "county"])[fill_cols].ffill()
    md = md.drop_duplicates(subset=["contractid", "planid", "state", "county"], keep="first")

    # Merge Part D info to Part C info
    out = ma.merge(md, on=["contractid", "planid", "state", "county"], how="outer")
    out["year"] = y
    return out

In [7]:
final_landscape_2012 = mapd_clean_merge_norecast(ma_data=ma_data, mapd_data=mapd_data, y=2012)
print(final_landscape_2012.shape)

(75416, 11)


## Penetration Data

In [8]:
from functions import MA_DATA_DIR

def read_penetration_safe(path: str | Path) -> pd.DataFrame:
    col_names = [
        "state", "county", "fips_state", "fips_cnty", "fips",
        "ssa_state", "ssa_cnty", "ssa", "eligibles", "enrolled", "penetration",
    ]
    df = pd.read_csv(
        path,
        skiprows=1,
        names=col_names,
        na_values=["", "NA", "*", "-", "--", "UK"],  # ✅ treat UK as missing
        dtype={  # ✅ read as string first so nothing crashes
            "state": "string",
            "county": "string",
            "fips_state": "string",
            "fips_cnty": "string",
            "fips": "string",
            "ssa_state": "string",
            "ssa_cnty": "string",
            "ssa": "string",
            "eligibles": "string",
            "enrolled": "string",
            "penetration": "string",
        },
        encoding="latin1",
        encoding_errors="replace",
        low_memory=False,
    )

    # ✅ coerce ID-ish numeric columns safely
    for c in ["fips_state", "fips_cnty", "fips", "ssa_state", "ssa_cnty", "ssa"]:
        df[c] = pd.to_numeric(df[c].str.replace(",", "", regex=False), errors="coerce")

    # ✅ parse numeric columns (handles commas and %)
    for c in ["eligibles", "enrolled", "penetration"]:
        df[c] = pd.to_numeric(
            df[c].astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
            errors="coerce",
        )

    return df

def load_month_pen_safe(m: str, y: int) -> pd.DataFrame:
    path = MA_DATA_DIR / "ma" / "penetration" / "Extracted Data" / f"State_County_Penetration_MA_{y}_{m}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing penetration file: {path}")
    df = read_penetration_safe(path)
    df["month"] = int(m)
    df["year"] = y
    return df

In [9]:
def agg_penetration(group):
    n_elig = group['eligibles'].notna().sum()
    n_enrol = group['enrolled'].notna().sum()
    elig_vals = group['eligibles'].dropna()
    enrol_vals = group['enrolled'].dropna()

    return pd.Series({
        'n_elig': n_elig,
        'n_enrol': n_enrol,
        'avg_eligibles': elig_vals.mean() if n_elig > 0 else np.nan,
        'first_eligibles': elig_vals.iloc[0] if n_elig > 0 else np.nan,
        'last_eligibles': elig_vals.iloc[-1] if n_elig > 0 else np.nan,
        'avg_enrolled': enrol_vals.mean() if n_enrol > 0 else np.nan,
        'sd_enrolled': enrol_vals.std() if n_enrol > 1 else np.nan,
        'min_enrolled': enrol_vals.min() if n_enrol > 0 else np.nan,
        'max_enrolled': enrol_vals.max() if n_enrol > 0 else np.nan,
        'first_enrolled': enrol_vals.iloc[0] if n_enrol > 0 else np.nan,
        'last_enrolled': enrol_vals.iloc[-1] if n_enrol > 0 else np.nan,
        'ssa': group['ssa'].iloc[-1]
    })

In [10]:
ma_penetration = pd.concat([load_month_pen_safe(m, y) for m in monthlist], ignore_index=True)

# Sort and fill fips
ma_penetration = ma_penetration.sort_values(['state', 'county', 'month'])
ma_penetration['fips'] = ma_penetration.groupby(['state', 'county'])['fips'].transform(lambda x: x.ffill().bfill())

pen_2012 = ma_penetration.groupby(['fips', 'state', 'county', 'year']).apply(agg_penetration).reset_index()
print(f"Penetration data shape: {pen_2012.shape}")

Penetration data shape: (3224, 16)


/tmp/ipykernel_220508/4102773071.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pen_2012 = ma_penetration.groupby(['fips', 'state', 'county', 'year']).apply(agg_penetration).reset_index()


## Rebate Data

In [11]:
import sys
!{sys.executable} -m pip install openpyxl


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
from functions import MA_DATA_DIR

def parse_number_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace(r"[^\d\.\-]+", "", regex=True)
    return pd.to_numeric(s, errors="coerce")

PAY_DIR = MA_DATA_DIR / "ma" / "cms-payment" / "2012"

# ✅ pick the files robustly
partc_candidates = sorted(PAY_DIR.glob("*PartC*PlanLevel*.xls*")) + sorted(PAY_DIR.glob("*PartC*.xls*"))
partd_candidates = sorted(PAY_DIR.glob("*PartD*Plans*.xls*")) + sorted(PAY_DIR.glob("*PartD*.xls*"))

if len(partc_candidates) == 0:
    raise FileNotFoundError(f"No Part C payment file found in {PAY_DIR}")
if len(partd_candidates) == 0:
    raise FileNotFoundError(f"No Part D payment file found in {PAY_DIR}")

ma_path_a = partc_candidates[0]
ma_path_b = partd_candidates[0]

print("Using Part C file:", ma_path_a)
print("Using Part D file:", ma_path_b)

risk_rebate_a = pd.read_excel(
    ma_path_a,
    sheet_name=0,
    usecols="A:H",
    skiprows=3,
    nrows=3157 - 4 + 1,
    header=None,
    names=[
        "contractid","planid","contract_name","plan_type",
        "riskscore_partc","payment_partc","rebate_partc","msa_deposit_partc"
    ],
)

risk_rebate_b = pd.read_excel(
    ma_path_b,
    sheet_name=0,
    usecols="A:H",
    skiprows=3,
    nrows=4417 - 4 + 1,
    header=None,
    names=[
        "contractid","planid","contract_name","plan_type",
        "directsubsidy_partd","riskscore_partd","reinsurance_partd","costsharing_partd"
    ],
)

for col in ["riskscore_partc", "payment_partc", "rebate_partc"]:
    risk_rebate_a[col] = parse_number_series(risk_rebate_a[col])

risk_rebate_a["planid"] = pd.to_numeric(risk_rebate_a["planid"], errors="coerce")
risk_rebate_a["year"] = 2012

risk_rebate_a = risk_rebate_a[
    ["contractid", "planid", "contract_name", "plan_type",
     "riskscore_partc", "payment_partc", "rebate_partc", "year"]
]

for col in ["directsubsidy_partd", "reinsurance_partd", "costsharing_partd"]:
    risk_rebate_b[col] = parse_number_series(risk_rebate_b[col])

risk_rebate_b["payment_partd"] = (
    risk_rebate_b["directsubsidy_partd"]
    + risk_rebate_b["reinsurance_partd"]
    + risk_rebate_b["costsharing_partd"]
)

risk_rebate_b["planid"] = pd.to_numeric(risk_rebate_b["planid"], errors="coerce")

risk_rebate_b = risk_rebate_b[
    ["contractid", "planid", "payment_partd",
     "directsubsidy_partd", "reinsurance_partd", "costsharing_partd",
     "riskscore_partd"]
]

for df in [risk_rebate_a, risk_rebate_b]:
    df["contractid"] = df["contractid"].astype("string").str.strip().str.upper()
    df["planid"] = pd.to_numeric(df["planid"], errors="coerce").astype("Int64")

final_risk_rebate = risk_rebate_a.merge(
    risk_rebate_b,
    on=["contractid", "planid"],
    how="left"
)

final_risk_rebate = risk_rebate_a.merge(
    risk_rebate_b,
    on=["contractid", "planid"],
    how="left"
)

print(f"Rebate data shape: {final_risk_rebate.shape}")

Using Part C file: /home/gdanzil/econ470/a0/work/ma-data/ma/cms-payment/2012/2012PartCPlanLevel.xlsx
Using Part D file: /home/gdanzil/econ470/a0/work/ma-data/ma/cms-payment/2012/2012PartDPlans.xlsx
Rebate data shape: (2754, 13)


## FFS Costs Data

In [13]:
from functions import MA_DATA_DIR
from pathlib import Path

FFS_DIR = MA_DATA_DIR / "ffs-costs" / "Extracted Data" / "Aged Only"

# list what you've actually got (helpful)
print("FFS_DIR:", FFS_DIR)
print("Some files:", [p.name for p in sorted(FFS_DIR.glob("*"))][:25])

candidates = []
for pat in ["*12*.csv", "*2012*.csv", "*aged12*.csv", "*FFS*12*.csv", "*Aged*12*.csv", "*.csv"]:
    candidates = sorted(FFS_DIR.glob(pat))
    if candidates:
        break

ffs_path = candidates[0]
print("Using FFS file:", ffs_path)

ffs_col_names = ["ssa", "state", "county_name", "parta_enroll",
                 "parta_reimb", "parta_percap", "parta_reimb_unadj",
                 "parta_percap_unadj", "parta_ime", "parta_dsh",
                 "parta_gme", "partb_enroll",
                 "partb_reimb", "partb_percap",
                 "mean_risk"]

ffs_data = pd.read_csv(
    ffs_path,
    skiprows=2,
    names=ffs_col_names,
    na_values=['*', '.'],
    usecols=range(15),
    encoding="latin1",
    encoding_errors="replace",
)

ffscosts_2012 = ffs_data[['ssa', 'state', 'county_name', 'parta_enroll', 'parta_reimb',
                          'partb_enroll', 'partb_reimb', 'mean_risk']].copy()
ffscosts_2012['year'] = 2012
ffscosts_2012['ssa'] = pd.to_numeric(ffscosts_2012['ssa'], errors='coerce')

for col in ['parta_enroll', 'parta_reimb', 'partb_enroll', 'partb_reimb', 'mean_risk']:
    ffscosts_2012[col] = pd.to_numeric(
        ffscosts_2012[col].astype(str).str.replace(r'[^\d.-]', '', regex=True),
        errors='coerce'
    )

print(f"FFS costs data shape: {ffscosts_2012.shape}")

FFS_DIR: /home/gdanzil/econ470/a0/work/ma-data/ffs-costs/Extracted Data/Aged Only
Some files: ['AGED06.csv', 'AGED08.csv', 'Aged07.csv', 'FFS15.xlsx', 'FFS16.xlsx', 'FFS17_AdvancedNotice.xlsx', 'FFS18.xlsx', 'aged09.csv', 'aged10.csv', 'aged11.csv', 'aged12.csv', 'aged13.csv', 'aged14.csv']
Using FFS file: /home/gdanzil/econ470/a0/work/ma-data/ffs-costs/Extracted Data/Aged Only/aged12.csv
FFS costs data shape: (3226, 9)


## Star Ratings

In [15]:
def parse_number_like_readr(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return float("nan")
    s = str(x).strip()
    if s == "":
        return float("nan")
    s = s.replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else float("nan")


ma_path_a = (
    "../ma-data/ma/star-ratings/Extracted Star Ratings/Part C 2012 Fall/"
    "2012_Part_C_Report_Card_Master_Table_2011_11_01_Domain.csv"
)

rating_vars_2012 = [
    "contractid",
    "org_type",
    "contract_name",
    "org_marketing",
    "breastcancer_screen",
    "rectalcancer_screen",
    "cv_diab_cholscreen",
    "glaucoma_test",
    "monitoring",
    "flu_vaccine",
    "pn_vaccine",
    "physical_health",
    "mental_health",
    "osteo_test",
    "physical_monitor",
    "primaryaccess",
    "osteo_manage",
    "diab_healthy",
    "bloodpressure",
    "ra_manage",
    "copd_test",
    "bladder",
    "falling",
    "nodelays",
    "doctor_communicate",
    "carequickly",
    "customer_service",
    "overallrating_care",
    "overallrating_plan",
    "complaints_plan",
    "appeals_timely",
    "appeals_review",
    "leave_plan",
    "audit_problems",
    "hold_times",
    "info_accuracy",
    "ttyt_available"
]

star_data_a = pd.read_csv(
    ma_path_a,
    skiprows=4,
    names=rating_vars_2012,  
    header=None,
    na_values=["", "NA", "*"],
    keep_default_na=True,
    encoding="latin1",
)

exclude_cols = {"contractid", "org_type", "contract_name", "org_marketing"}
cols_to_parse = [c for c in star_data_a.columns if c not in exclude_cols]
for c in cols_to_parse:
    star_data_a[c] = star_data_a[c].map(parse_number_like_readr).astype("float64")


ma_path_b = (
    "../ma-data/ma/star-ratings/Extracted Star Ratings/Part C 2012 Fall/"
    "2012_Part_C_Report_Card_Master_Table_2011_11_01_Summary.csv"
)
star_data_b = pd.read_csv(
    ma_path_b,
    skiprows=2,
    names=["contractid", "org_type", "contract_name", "org_marketing", "partc_score"],
    header=None,
    na_values=["", "NA", "*"],
    keep_default_na=True,
    encoding="latin1",
)

# mutate(new_contract=..., partc_score=...) %>% select(...) %>% mutate(partcd_score=NA_real_)
star_data_b = star_data_b.assign(
    new_contract=lambda d: (d["partc_score"] == "Plan too new to be measured").astype("int64"),
)

star_data_b["partc_score"] = star_data_b.apply(
    lambda r: float("nan") if r["new_contract"] == 1 else parse_number_like_readr(r["partc_score"]),
    axis=1
).astype("float64")

star_data_b = (
    star_data_b
    .loc[:, ["contractid", "new_contract", "partc_score"]]
    .assign(partcd_score=pd.NA)   # NA_real_ equivalent
)

final_star_ratings = (
    star_data_a
    .drop(columns=["contract_name", "org_type", "org_marketing"], errors="ignore")
    .merge(star_data_b, on="contractid", how="left")   # left_join
    .assign(year=2012)
)

print(f"Star Ratings data shape: {final_star_ratings.shape}")     

Star Ratings data shape: (567, 38)


## Benchmark Data

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from functions import MA_DATA_DIR

BENCH_DIR = MA_DATA_DIR / "ma" / "benchmarks"

candidates = list(BENCH_DIR.rglob("CountyRate2012.csv"))
if len(candidates) == 0:
    # fallback: any county rate file for 2010 (in case name slightly differs)
    candidates = [p for p in BENCH_DIR.rglob("*.csv") if ("2012" in p.name and "CountyRate" in p.name)]

if len(candidates) == 0:
    raise FileNotFoundError(
        f"Couldn't find CountyRate2012.csv under {BENCH_DIR}. "
        f"Try: !find '{BENCH_DIR}' -maxdepth 5 -type f | head"
    )

bench_path = sorted(candidates)[0]
print("Using benchmark file:", bench_path)

# Read CSV (skip first 10 rows, specify column names)
bench_data = pd.read_csv(
    bench_path,
    skiprows=10,
    names=[
        "ssa", "state", "county_name", "aged_parta",
        "aged_partb", "disabled_parta", "disabled_partb",
        "esrd_ab", "risk_ab"
    ],
    encoding="latin1",
    encoding_errors="replace",
)

# Select columns and add numeric NA columns
final_benchmark = (
    bench_data[["ssa", "aged_parta", "aged_partb", "risk_ab"]]
    .assign(
        risk_star5=np.nan,
        risk_star45=np.nan,
        risk_star4=np.nan,
        risk_star35=np.nan,
        risk_star3=np.nan,
        risk_star25=np.nan,
        risk_bonus5=np.nan,
        risk_bonus35=np.nan,
        risk_bonus0=np.nan,
        year=2012
    )
)

print(final_benchmark.head())
print("Benchmark shape:", final_benchmark.shape)

Using benchmark file: /home/gdanzil/econ470/a0/work/ma-data/ma/benchmarks/ratebook2012/CountyRate2012.csv
     ssa aged_parta aged_partb   risk_ab  risk_star5  risk_star45  risk_star4  \
1010  AL     848.94     848.94  6,687.81         NaN          NaN         NaN   
1020  AL     778.87     778.87  6,687.81         NaN          NaN         NaN   
1030  AL     878.70     878.70  6,687.81         NaN          NaN         NaN   
1040  AL     855.90     855.90  6,687.81         NaN          NaN         NaN   
1050  AL     786.67     786.67  6,687.81         NaN          NaN         NaN   

      risk_star35  risk_star3  risk_star25  risk_bonus5  risk_bonus35  \
1010          NaN         NaN          NaN          NaN           NaN   
1020          NaN         NaN          NaN          NaN           NaN   
1030          NaN         NaN          NaN          NaN           NaN   
1040          NaN         NaN          NaN          NaN           NaN   
1050          NaN         NaN          NaN

## Merge Data

In [17]:
import pandas as pd
import numpy as np
import re

def na0(x):
    return 0 if pd.isna(x) else x

def parse_money(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "")
    s = re.sub(r"[^0-9\.\-]", "", s)  # strips $, %, spaces, etc
    return pd.to_numeric(s, errors="coerce")

def calc_basic_premium(row):
    rebate_partc = na0(row.get("rebate_partc"))
    premium = row.get("premium")
    premium_partc = row.get("premium_partc")

    if rebate_partc > 0:
        return 0
    elif row.get("partd") == "No" and pd.notna(premium) and pd.isna(premium_partc):
        return premium
    else:
        return premium_partc

def calc_bid(row):
    rebate = na0(row.get("rebate_partc"))
    basic_premium = na0(row.get("basic_premium"))
    payment_partc = row.get("payment_partc")
    riskscore_partc = row.get("riskscore_partc")

    if pd.isna(payment_partc) or pd.isna(riskscore_partc) or riskscore_partc == 0:
        return np.nan

    if rebate == 0 and basic_premium > 0:
        return (payment_partc + basic_premium) / riskscore_partc

    return payment_partc / riskscore_partc

def normalize_merge_keys(df, *,
                         contract_col="contractid",
                         plan_col="planid",
                         state_col=None,
                         county_col=None):
    if contract_col in df.columns:
        df[contract_col] = df[contract_col].astype("string").str.strip().str.upper()

    if plan_col in df.columns:
        df[plan_col] = (
            pd.to_numeric(df[plan_col].astype(str).str.replace(r"\.0$", "", regex=True), errors="coerce")
            .astype("Int64")
        )

    if state_col and state_col in df.columns:
        df[state_col] = df[state_col].astype("string").str.strip().str.lower()

    if county_col and county_col in df.columns:
        df[county_col] = df[county_col].astype("string").str.strip().str.lower()

    return df

ma_2012 = plan_data_2012.merge(
    service_data_2012[["contractid", "fips"]],
    on=["contractid", "fips"],
    how="inner"
)

excluded_states = ["VI", "PR", "MP", "GU", "AS", ""]
ma_2012 = ma_2012[
    (~ma_2012["state"].isin(excluded_states)) &
    (ma_2012["snp"] == "No") &
    ((ma_2012["planid"] < 800) | (ma_2012["planid"] >= 900)) &
    (ma_2012["planid"].notna()) &
    (ma_2012["fips"].notna())
].copy()


pen_2012_join = pen_2012.copy()
pen_2012_join = pen_2012_join.rename(columns={"state": "state_long", "county": "county_long"})
pen_2012_join["state_long"] = pen_2012_join["state_long"].astype("string").str.lower()

pen_2012_join["ncount"] = pen_2012_join.groupby("fips")["fips"].transform("count")
pen_2012_join = pen_2012_join[pen_2012_join["ncount"] == 1].drop(columns=["ncount"])

ma_2012 = ma_2012.merge(pen_2012_join, on="fips", how="left", suffixes=("", "_pen"))


state_2012 = (
    ma_2012.loc[ma_2012["state_long"].notna(), ["state", "state_long"]]
    .drop_duplicates(subset=["state"], keep="last")
    .rename(columns={"state_long": "state_name"})
)

full_2012 = ma_2012.merge(state_2012, on="state", how="left")


landscape_2012_join = final_landscape_2012.copy()

landscape_2012_join["state"] = landscape_2012_join["state"].astype("string").str.strip().str.lower()
landscape_2012_join["county"] = landscape_2012_join["county"].astype("string").str.strip().str.lower()

normalize_merge_keys(full_2012, county_col="county")
normalize_merge_keys(landscape_2012_join, state_col="state", county_col="county")

full_2012["state_name"] = full_2012["state_name"].astype("string").str.strip().str.lower()

full_2012 = full_2012.merge(
    landscape_2012_join,
    left_on=["contractid", "planid", "state_name", "county"],
    right_on=["contractid", "planid", "state", "county"],
    how="left",
    suffixes=("", "_land")
)

rebate_2012_join = final_risk_rebate.drop(columns=["contract_name", "plan_type"], errors="ignore").copy()
normalize_merge_keys(full_2012)
normalize_merge_keys(rebate_2012_join)

full_2012 = full_2012.merge(rebate_2012_join, on=["contractid", "planid"], how="left")

for c in ["premium", "premium_partc", "rebate_partc", "payment_partc", "riskscore_partc"]:
    if c in full_2012.columns:
        full_2012[c] = full_2012[c].map(parse_money)

full_2012["basic_premium"] = full_2012.apply(calc_basic_premium, axis=1)
full_2012["bid"] = full_2012.apply(calc_bid, axis=1)

drop_cols = [c for c in ["contract_name", "org_type", "org_marketing"] if c in final_star_ratings.columns]
star_2012_join = final_star_ratings.drop(columns=drop_cols, errors="ignore").copy()

normalize_merge_keys(full_2012, plan_col="planid")
normalize_merge_keys(star_2012_join, plan_col=None)

full_2012 = full_2012.merge(star_2012_join, on="contractid", how="left", suffixes=("", "_star"))

for col in ["partc_score", "partcd_score"]:
    if col in full_2012.columns:
        full_2012[col] = pd.to_numeric(full_2012[col], errors="coerce")

full_2012["Star_Rating"] = np.where(
    full_2012["partd"] == "No",
    full_2012["partc_score"],
    np.where(full_2012["partcd_score"].isna(), full_2012["partc_score"], full_2012["partcd_score"])
)

bench_2012_join = final_benchmark.loc[final_benchmark["ssa"].notna()].copy()
full_2012["ssa"] = pd.to_numeric(full_2012["ssa"], errors="coerce")
bench_2012_join["ssa"] = pd.to_numeric(bench_2012_join["ssa"], errors="coerce")

full_2012 = full_2012.merge(bench_2012_join, on="ssa", how="left", suffixes=("", "_bench"))

print("Final merged data shape:", full_2012.shape)

Final merged data shape: (88599, 106)


In [18]:
# SAVE TO CSV
output_path = "../Data/output/data-2012.csv"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
full_2012.to_csv(output_path, index=False)
print(f"Data saved to {output_path}")

Data saved to ../Data/output/data-2012.csv
